In [1]:
import sys
sys.path.insert(0, '..')

In [2]:
from src.config import get_spark, BRONZE_PRODUCTS, SILVER_PRODUCTS
from delta.tables import DeltaTable

In [3]:
spark = get_spark()

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ee197d0d-573f-4489-bfae-ab317500f1ad;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 367ms :: artifacts dl 19ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 

In [4]:
print('=== BRONZE ===')
df_b = spark.read.format('delta').load(str(BRONZE_PRODUCTS))
df_b.show(5)
df_b.printSchema()
print('Row count:', df_b.count())
print('=== SILVER ===')
df_s = spark.read.format('delta').load(str(SILVER_PRODUCTS))
df_s.show(5)
df_s.printSchema()
print('Row count:', df_s.count())

=== BRONZE ===


26/06/26 15:21:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/06/26 15:21:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------------+---------------+--------------------+--------+--------------------+--------------------+------------+----------------+--------------------+--------------------+----------------+--------+------------------+------------------+-----------+-------------+-----------------+--------------------+
|         code|last_modified_t|        product_name|  brands|       categories_en|           labels_en|countries_en|nutriscore_grade|    ingredients_tags|      food_groups_en|energy-kcal_100g|fat_100g|saturated-fat_100g|carbohydrates_100g|sugars_100g|proteins_100g|        salt_100g|         ingest_at_t|
+-------------+---------------+--------------------+--------+--------------------+--------------------+------------+----------------+--------------------+--------------------+----------------+--------+------------------+------------------+-----------+-------------+-----------------+--------------------+
|3263850578311|     1774306130|Génoises fourrage...|Franprix|Snacks,Sweet snac...|   

Row count: 4475958
=== SILVER ===


+--------+---------------+--------------------+-------------+---------+--------------+--------------------+--------------------+----------------+--------------------+----------------+--------+------------------+------------------+-----------+-------------+---------+
|    code|last_modified_t|        product_name|       brands|labels_en|  countries_en|      food_groups_en|       categories_en|nutriscore_grade|    ingredients_tags|energy-kcal_100g|fat_100g|saturated-fat_100g|carbohydrates_100g|sugars_100g|proteins_100g|salt_100g|
+--------+---------------+--------------------+-------------+---------+--------------+--------------------+--------------------+----------------+--------------------+----------------+--------+------------------+------------------+-----------+-------------+---------+
|00000010|     1750695017|                 xxx|          xxx|   it:xxx|       Germany|Beverages,Sweeten...|Beverages and bev...|            NULL|[wheat-flour, cer...|            NULL|    NULL|       

[Stage 35:===========================================>              (3 + 1) / 4]

Row count: 4136190


In [6]:
DeltaTable.forPath(spark, str(BRONZE_PRODUCTS)).history().show(truncate=False)

+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                      |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-06-26 14:48:54|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |false        |{

In [7]:
DeltaTable.forPath(spark, str(SILVER_PRODUCTS)).history().show(truncate=False)

+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                      |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-06-26 14:54:22|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |false        |{

In [8]:
spark.stop()